# Classical versus learned hedging (display from committed artifacts)

Headline cell: frozen `BenchConfig` (Heston ATM 1M, **20 bp full-rate** cost) and
`bench/classical_best.json`. Historical 4×5 comparison and the locked holdout
are already in `REPORT.md` §3; this notebook **loads** the JSON mirror under
`submission/data/` — it does not re-score paths or invent Sharpe edges.

Re-run writers (optional, slow): `archive/scripts/comparison_study.py`,
`scripts/holdout_eval.py` (holdout only, once).


In [ ]:
import json
import pathlib
import sys

import matplotlib.pyplot as plt
import polars as pl

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

from hedging import bench as B

cfg = B.BenchConfig()
baseline = B.load_classical_best()
print("BenchConfig:", cfg.model, "cost=", cfg.cost, "n_steps=", cfg.n_steps)
print("classical_best.json:", baseline)

## Fixed-hyper edge map (`comparison_study.py --fixed-hyper`)

Same table as `REPORT.md` §3.1 (model × cost in bp). Edge = mean validation
Sharpe (learned − classical), averaged over training seeds.


In [ ]:
EDGE_JSON = ROOT / "submission" / "data" / "comparison" / "edge_map_fixed.json"
EDGE_PNG = ROOT / "submission" / "data" / "comparison" / "edge_map_fixed.png"
cells = json.loads(EDGE_JSON.read_text())
edge = pl.DataFrame(cells).with_columns((pl.col("cost") * 1e4).alias("cost_bp"))
edge.select(
    "model",
    "cost_bp",
    "edge_mean",
    "edge_min",
    "win_rate",
    "deep_sharpe",
    "classical_sharpe",
    "deep_std",
    "classical_std",
).sort("model", "cost_bp")

In [ ]:
if EDGE_PNG.is_file():
    plt.figure(figsize=(8, 5))
    plt.imshow(plt.imread(EDGE_PNG))
    plt.axis("off")
    plt.title("Committed edge map (see REPORT §3.1)")
    plt.show()

## Locked holdout (`scripts/holdout_eval.py`)

Single run on the five holdout seeds; summary matches `REPORT.md` §3.4.


In [ ]:
HOLDOUT_LOG = ROOT / "submission" / "data" / "holdout_history.jsonl"
last = HOLDOUT_LOG.read_text().strip().splitlines()[-1]
holdout = json.loads(last)
pl.DataFrame([holdout]).select("edge_mean", "edge_std", "edge_min", "win_rate", "n_seeds", "timestamp")

### Reading (from the report, not re-derived here)

At **20 bp** on the frozen Heston cell the fixed-hyper comparison is flat
(`heston` edge about +0.014 in the map above). The holdout mean edge (+0.0339)
is the same order of magnitude — a small win, not a overturning of the classical
band story. High-cost Sharpe edges in the map are premium-harvesting artefacts;
see `REPORT.md` §3.1 regime 3.
